In [ ]:
import random

import pandas as pd
from IPython.display import VimeoVideo
from pymongo import MongoClient
from teaching_tools.ab_test.reset import Reset

r = Reset("192.252.53.2")
r.reset_database()


In [ ]:
VimeoVideo("742770800", h="ce17b05c51", width=600)


In [ ]:
host = "192.252.53.2"


In [ ]:
client = MongoClient(host=host, port=27017)
ds_app = client["wqu-abtest"]["ds-applicants"]
print("client:", type(client))
print("ds_app:", type(ds_app))


In [ ]:
VimeoVideo("734130688", h="637d2529dc", width=600)


In [ ]:
# How many applicants complete admissions quiz?

result = ds_app.aggregate(
    [{
        "$group": {
            "_id": "$admissionsQuiz",
            "count": {"$count": {}}
        }
    }
] )

for r in result:
    if r["_id"] == "incomplete":
        incomplete = r["count"]
    else:
        complete = r["count"]

print("Completed quiz:", complete)
print("Did not complete quiz:", incomplete)


In [ ]:
VimeoVideo("734130558", h="b06dabae44", width=600)


In [ ]:
total = complete + incomplete
prop_incomplete = incomplete / total

print(
    "Proportion of users who don't complete admissions quiz:", round(prop_incomplete, 2)
)


In [ ]:
VimeoVideo("734130136", h="e1c88a9ecd", width=600)


In [ ]:
VimeoVideo("734131639", h="7e9aac1e60", width=600)


In [ ]:
null_hypothesis = """
There is no relationship between receiving an email and completing the admissions quiz.
Sending an email to 'no-quiz applicants' does not increase the rate of completion.
"""

alternate_hypothesis = """
There is a relationship between receiving an email and completing the admissions quiz.
Sending an email to 'no-quiz applicants' does increase the rate of completion.
"""

print("Null Hypothesis:", null_hypothesis)
print("Alternate Hypothesis:", alternate_hypothesis)


In [ ]:
VimeoVideo("734136019", h="227630f2d2", width=600)


In [ ]:
def find_by_date(collection, date_string):
    """Find records in a PyMongo Collection created on a given date.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection in which to search for documents.
    date_string : str
        Date to query. Format must be '%Y-%m-%d', e.g. '2022-06-28'.

    Returns
    -------
    observations : list
        Result of query. List of documents (dictionaries).
    """
    collection = ds_app
    date_string = "2022-05-04"
    # Convert `date_string` to datetime object
    start = start = pd.to_datetime(date_string, format="%Y-%m-%d")
    # Offset `start` by 1 day
    end = end = start + pd.DateOffset(days=1)
    # Create PyMongo query for no-quiz applicants b/t `start` and `end`
    query = {"createdAt": {"$gte": start, "$lt": end}, "admissionsQuiz": "incomplete"}
    # Query collection, get result
    result = collection.find(query)
    # Convert `result` to list
    observations = list(result)
    return observations


In [ ]:
VimeoVideo("734135947", h="172e5d7e19", width=600)


In [ ]:
observations = find_by_date(collection=ds_app, date_string="2022-05-15")

print("observations type:", type(observations))
print("observations len:", len(observations))
observations[0]


In [ ]:
VimeoVideo("734134939", h="d7b409da4b", width=600)


In [ ]:
def assign_to_groups(observations):
    """Randomly assigns observations to control and treatment groups.

    Parameters
    ----------
    observations : list or pymongo.cursor.Cursor
        List of users to assign to groups.

    Returns
    -------
    observations : list
        List of documents from `observations` with two additional keys:
        `inExperiment` and `group`.
    """
   
    # Shuffle `observations`
    random.seed(42)
    random.shuffle(observations)

    # Get index position of item at observations halfway point
    idx = len(observations) // 2

    # Assign first half of observations to control group
    for doc in observations[:idx]:
        doc["inExperiment"] = True
        doc["group"] = "no email (control)"

    # Assign second half of observations to treatment group
    for doc in observations[idx:]:
        doc["inExperiment"] = True
        doc["group"] = "email (treatment)"

    return observations

observations_assigned = assign_to_groups(observations)

print("observations_assigned type:", type(observations_assigned))
print("observations_assigned len:", len(observations_assigned))
observations_assigned[0]


In [ ]:
VimeoVideo("734137698", h="87610a6a1c", width=600)


In [ ]:
def export_treatment_emails(observations_assigned, directory="."):
    """Creates CSV file with email addresses of observations in treatment group.

    CSV file name will include today's date, e.g. `'2022-06-28_ab-test.csv'`,
    and a `'tag'` column where every row will be 'ab-test'.

    Parameters
    ----------
    observations_assigned : list
        Observations with group assignment.
    directory : str, default='.'
        Location for saved CSV file.

    Returns
    -------
    None
    """
    # Put `observations_assigned` docs into DataFrame
    df = pd.DataFrame(observations_assigned)

    # Add `"tag"` column
    df["tag"] = "ab-test"
    
    # Create mask for treatment group only
    mask = df["group"] == "email (treatment)"
    
    # Create filename with date
    date_string = pd.Timestamp.now().strftime(format="%Y-%m-%d")
    filename = directory + "/" + date_string + "_ab-test.csv"
    print(filename)
    
    # Save DataFrame to directory (email and tag columns only)
    df[mask][["email", "tag"]].to_csv(filename, index=False)
    

export_treatment_emails(observations_assigned=observations_assigned)


In [ ]:
VimeoVideo("734137546", h="e07cebf91e", width=600)


In [ ]:
updated_applicant = observations_assigned[0]
# Use square brackets to access the dictionary key
applicant_id = updated_applicant["_id"]

print("applicant type:", type(updated_applicant))
print(updated_applicant)
print()
print("applicant_id type:", type(applicant_id))
print(applicant_id)


In [ ]:
VimeoVideo("734137409", h="5ea2eaf949", width=600)


In [ ]:
# Find original record for `applicant_id`
ds_app.find_one({"_id": applicant_id})


In [ ]:
VimeoVideo("734141207", h="afe52c4d42", width=600)


In [ ]:
result = ds_app.update_one(
    filter={"_id": applicant_id},
    update={"$set": updated_applicant}
)
print("result type:", type(result))


In [ ]:
VimeoVideo("734142198", h="eabd16f09e", width=600)


In [ ]:
# Access methods and attributes using `dir`
dir(result)
# Access `raw_result` attribute
result.raw_result


In [ ]:
VimeoVideo("734147474", h="4e38b07a71", width=600)


In [ ]:
def update_applicants(collection, observations_assigned):
    """Update applicant documents in collection.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection in which documents will be updated.

    observations_assigned : list
        Documents that will be used to update collection

    Returns
    -------
    transaction_result : dict
        Status of update operation, including number of documents
        and number of documents modified.
    """
    # Initialize counters
    n = 0
    n_modified = 0

    # Iterate through applicants
    for doc in observations_assigned:
        # Update doc
        result = collection.update_one(
            filter={"_id": doc["_id"]},
            update={"$set": doc})
    
        # create result
        n += result.matched_count
        n_modified += result.modified_count

    # Create results
    transaction_result = {"n": n, "nModified": n_modified}
   
    return transaction_result


In [ ]:
result = update_applicants(ds_app, observations_assigned)
print("result type:", type(result))
result


In [ ]:
VimeoVideo("734133492", h="a0f97831a1", width=600)


In [ ]:
VimeoVideo("734133039", h="070a04dd1c", width=600)


In [ ]:
class MongoRepository:
    """Repository class for interacting with MongoDB database.

    Parameters
    ----------
    client : `pymongo.MongoClient`
        By default, `MongoClient(host=host, port=27017)`.
    db : str
        By default, `'wqu-abtest'`.
    collection : str
        By default, `'ds-applicants'`.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        All data will be extracted from and loaded to this collection.
    """

    # Task 7.2.14
    def __init__(
        self,
        client=MongoClient(host="192.252.53.2", port=27017),
        db="wqu-abtest",
        collection="ds-applicants"
    ):
        self.collection = client[db][collection]

    # Task 7.2.17
    def find_by_date(self, date_string):
        # Find start datetime
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
        # Calculate the end datetime
        end = start + pd.DateOffset(days=1)
        # Build PyMongo query
        query = {"createdAt": {"$gte": start, "$lt": end}, "admissionsQuiz": "incomplete"}
        # Send query to collection, get results
        result = self.collection.find(query)
        # Put results into list
        observations = list(result)
        return observations
        # Task 7.2.18
    def update_applicants(self, observations_assigned):
        # Initialize counters
        n = 0
        n_modified = 0
    
        # Iterate through applicants
        for doc in observations_assigned:
            # Update doc
            result = self.collection.update_one(
                filter={"_id": doc["_id"]},
                update={"$set": doc}
        )
        
            # create result
            n += result.matched_count
            n_modified += result.modified_count
    
        # Create results
        transaction_result = {"n": n, "nModified": n_modified}
   
        return transaction_result
        # Task 7.2.19
    def assign_to_groups(self, date_string):
        # Get observations
        observations = self.find_by_date(date_string)
        # Shuffle `observations`
        random.seed(42)
        random.shuffle(observations)
    
        # Get index position of item at observations halfway point
        idx = len(observations) // 2
    
        # Assign first half of observations to control group
        for doc in observations[:idx]:
            doc["inExperiment"] = True
            doc["group"] = "no email (control)"
    
        # Assign second half of observations to treatment group
        for doc in observations[idx:]:
            doc["inExperiment"] = True
            doc["group"] = "email (treatment)"

        # Update collection
        result = self.update_applicants(observations)
        return result


In [ ]:
VimeoVideo("734150578", h="2caaa53d03", width=600)


In [ ]:
repo = MongoRepository()
print("repo type:", type(repo))
repo


In [ ]:
VimeoVideo("734150427", h="f9c9433ff6", width=600)


In [ ]:
c_test = repo.collection
print("c_test type:", type(c_test))
c_test


In [ ]:
VimeoVideo("734150075", h="82f7810cd0", width=600)


In [ ]:
may_15_users = repo.find_by_date(date_string="2022-05-15")
print("may_15_users type", type(may_15_users))
print("may_15_users len", len(may_15_users))
may_15_users[:3]


In [ ]:
VimeoVideo("734149871", h="4db7c08002", width=600)


In [ ]:
result = repo.update_applicants(observations_assigned)
print("result type:", type(result))
result


In [ ]:
VimeoVideo("734149186", h="65f443159c", width=600)


In [ ]:
result = repo.assign_to_groups(date_string="2022-05-15")
print("result type:", type(result))
result


In [ ]:
VimeoVideo("734148753", h="2305068b6b", width=600)


In [ ]:
repo_test = MongoRepository()
submission = repo_test.assign_to_groups("2022-05-16")
